In [2]:
from ultralytics import YOLO
import easyocr
import cv2
import numpy as np

# Load models
helmet_model = YOLO(r'C:\Users\manis\OneDrive\Desktop\Visionguard\models\runs\detect\visionguard_helmet-4\weights\best.pt')
plate_model = YOLO(r'C:\Users\manis\OneDrive\Desktop\Visionguard\models\runs\detect\visionguard_plate\weights\best.pt')

# OCR engine
reader = easyocr.Reader(['en'])

def run_detection(image_path):
    img = cv2.imread(image_path)
    
    # Step 1: Helmet violation detection
    helmet_results = helmet_model(image_path)
    violations = []
    for box in helmet_results[0].boxes:
        cls = int(box.cls)
        conf = float(box.conf)
        label = helmet_results[0].names[cls]
        if label == 'Without Helmet':
            violations.append({'type': 'No Helmet', 'confidence': conf})
    
    # Step 2: Plate detection + OCR
    plate_results = plate_model(image_path)
    plates = []
    for box in plate_results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cropped_plate = img[y1:y2, x1:x2]
        ocr_result = reader.readtext(cropped_plate)
        if ocr_result:
            plate_text = ocr_result[0][1]
            plates.append(plate_text)
    
    # Step 3: Annotated output
    annotated = helmet_results[0].plot()
    cv2.imwrite('output_annotated.jpg', annotated)
    
    print("Violations:", violations)
    print("Plates:", plates)
    return violations, plates

# Run
run_detection(r'../data/sample_images/image9.png')


image 1/1 c:\Users\manis\OneDrive\Desktop\Visionguard\models\..\data\sample_images\image9.png: 448x640 1 With Helmet, 71.6ms
Speed: 3.8ms preprocess, 71.6ms inference, 2.2ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 c:\Users\manis\OneDrive\Desktop\Visionguard\models\..\data\sample_images\image9.png: 448x640 3 indian_licence_plates, 33.2ms
Speed: 3.8ms preprocess, 33.2ms inference, 2.8ms postprocess per image at shape (1, 3, 448, 640)
Violations: []
Plates: []


([], [])